In [3]:
from time import sleep, time
import requests
import os
import pandas as pd
import json

api_key = '97D5425D-C330-49D5-A048-11D7B089DD91'

file = open('../data/OpenAPI/dmobs.json', 'r', encoding='utf8')
dmobs = json.load(file)
file.close()
for item in dmobs['content']:
    if item is None or item['dmobscd'] != '1017310':
        continue
    dmobscd = item['dmobscd']
    site = f'../data/OpenAPI/dm/{dmobscd}'
    if not os.path.isdir(site):
        os.mkdir(site)
    
    for yr in range(2012, 2022+1):
        for mo in range(1, 12+1):
            tic = time()
            
            if yr == 2022 and mo == 8:
                break
            if mo in [1, 3, 5, 7, 8, 10, 12]:
                dd = 31
            elif mo in [4, 6, 9, 11]:
                dd = 30
            elif mo == 2 and yr%4 == 0:
                dd = 29
            else:
                dd = 28
            url = f'http://api.hrfco.go.kr/{api_key}/dam/list/10M/{dmobscd}/{yr}{mo:0>2}010000/{yr}{mo:0>2}{dd}2350.json'
            response = requests.get(url)
            response_json = response.json()
            if not (200 <= response.status_code < 300):
                print(response.status_code, dmobscd, yr, mo)
            df = pd.DataFrame(response_json['content'])[['ymdhm', 'swl', 'inf', 'sfw', 'ecpc', 'tototf']]
#             df = df.rename(columns={'rf': f'rf_{rfobscd}'})
            df['ymdhm'] = pd.to_datetime(df['ymdhm'])
            df.to_csv(f'{site}/dm_{yr}{mo:0>2}.csv', index=False)
            
            toc = time()
            
            sleep( max(0.1 - (toc - tic), 0) )